# EDA Project: Exploratory Analysis of House Prices
**Dataset:** Ames Housing — Kaggle `train.csv`  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, SciPy  
**Objective:** Explore house-sale prices and their relationships with house features. This project performs exploratory data analysis only; it does not train a machine-learning model.

> Run the notebook from the `House_Price_EDA_Project` folder. The CSV is expected at `dataset/train.csv`, and charts will be saved in `outputs/`.

## 1. Setup and first look
Load the libraries and inspect the dataset's size, columns, data types, and missing values.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "dataset" / "train.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Open Jupyter from the House_Price_EDA_Project folder."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())
df.info()

missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"Missing values": missing, "Missing %": missing_pct})
display(missing_summary[missing_summary["Missing values"] > 0])
print("Columns with missing values:", int((missing > 0).sum()))

## 2. Classify variables
Separate numerical and categorical columns. `OverallQual` is stored as a number, but it represents ordered quality levels, so it is ordinal. `ExterQual` and `KitchenQual` are also ordinal.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

ordinal_map = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"]
}

variable_rows = []
for col in df.columns:
    if col == "OverallQual":
        var_type = "Ordinal (numeric codes)"
    elif col in ordinal_map:
        var_type = "Ordinal (text categories)"
    elif col in numeric_cols:
        var_type = "Numerical"
    else:
        var_type = "Categorical"
    examples = df[col].dropna().unique()[:5].tolist()
    variable_rows.append({
        "Column": col,
        "Type": var_type,
        "Unique values": df[col].nunique(dropna=True),
        "Example values": str(examples)
    })

variable_summary = pd.DataFrame(variable_rows)
display(variable_summary.head(20))
print("Numerical columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Note: OverallQual is ordinal even though it is numeric.")

## 3. Descriptive statistics
Calculate centre and spread for the target price and two important area variables.

In [ ]:
display(df.describe().T)

for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col].dropna()
    print(f"\\n{col}")
    print("Mean:", round(s.mean(), 2))
    print("Median:", round(s.median(), 2))
    print("Mode:", s.mode().iloc[0])
    print("Standard deviation:", round(s.std(), 2))
    print("Variance:", round(s.var(), 2))
    print("Range:", round(s.max() - s.min(), 2))
    print("IQR:", round(s.quantile(.75) - s.quantile(.25), 2))
    print("Skewness:", round(s.skew(), 3))
    print("Kurtosis:", round(s.kurtosis(), 3))

print("\\nInterpretation: When the mean is greater than the median, a right tail / high-price values may be pulling the mean upward.")
display(df["Neighborhood"].value_counts().head(10).rename("Count").to_frame())
display(df["HouseStyle"].value_counts().rename("Count").to_frame())

## 4. Distributions
Use histograms, KDE curves, and boxplots to understand price distribution and compare prices across quality and neighbourhood groups.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice distribution")
axes[0].set_xlabel("Sale price ($)")
sns.boxplot(x=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice boxplot")
log_price = np.log1p(df["SalePrice"])
sns.histplot(log_price, kde=True, ax=axes[2])
axes[2].set_title("Log-transformed SalePrice")
axes[2].set_xlabel("log(1 + SalePrice)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("SalePrice skewness:", round(df["SalePrice"].skew(), 3))
print("Log SalePrice skewness:", round(log_price.skew(), 3))

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="OverallQual", y="SalePrice")
plt.title("Sale price by overall quality")
plt.xlabel("Overall quality (1–10)")
plt.ylabel("Sale price ($)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_by_quality.png", dpi=150, bbox_inches="tight")
plt.show()

neighborhood_order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=neighborhood_order)
plt.xticks(rotation=90)
plt.title("Sale price by neighborhood (ordered by median)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "price_by_neighborhood.png", dpi=150, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df, x="OverallQual", ax=axes[0])
axes[0].set_title("Count of houses by overall quality")
sns.countplot(data=df, x="HouseStyle", ax=axes[1])
axes[1].set_title("Count of houses by house style")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "category_counts.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Correlation and covariance
Pearson correlation measures the strength and direction of a linear relationship on a comparable -1 to +1 scale. Covariance depends on the variables' units.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr(method="pearson")
cov_matrix = numeric_df.cov()

top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False)
top10_features = top_corr.head(10).index.tolist()
top10_table = corr_matrix.loc[top10_features, "SalePrice"].sort_values(key=abs, ascending=False)
display(top10_table.rename("Correlation with SalePrice").to_frame())

for feature in top10_features[:3]:
    print(feature,
          "| covariance:", round(cov_matrix.loc[feature, "SalePrice"], 2),
          "| correlation:", round(corr_matrix.loc[feature, "SalePrice"], 3))

print("\\nExample: GarageArea and GarageCars may correlate because larger garages often fit more cars.")

## 6. Scatter plots and pair plot
Scatter plots help inspect how numerical features move with sale price and reveal unusual observations.

In [ ]:
for feature in ["GrLivArea", "TotalBsmtSF", "YearBuilt"]:
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df, x=feature, y="SalePrice", alpha=.65)
    plt.title(f"{feature} vs SalePrice")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"scatter_{feature}.png", dpi=150, bbox_inches="tight")
    plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", hue="OverallQual", palette="viridis", alpha=.75)
plt.title("Living area vs sale price, coloured by quality")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "living_area_vs_price_quality.png", dpi=150, bbox_inches="tight")
plt.show()

pair_cols = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]
pair = sns.pairplot(df[pair_cols].dropna(), diag_kind="kde", corner=True)
pair.fig.suptitle("Pair plot of selected numerical features", y=1.02)
pair.savefig(OUTPUT_DIR / "pairplot.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Correlation heatmap
Display a heatmap for the ten numerical features most strongly correlated with SalePrice, plus SalePrice itself.

In [ ]:
heatmap_cols = top10_features + ["SalePrice"]
corr_top = df[heatmap_cols].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool), k=1)

plt.figure(figsize=(11, 8))
sns.heatmap(corr_top, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True)
plt.title("Top features correlated with SalePrice")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Outlier detection
Compare the IQR rule and Z-score rule for `GrLivArea` and `SalePrice`. These methods flag unusual values; they do not automatically prove that a record is an error.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([.25, .75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

outlier_rows = []
outlier_masks = {}
for col in ["GrLivArea", "SalePrice"]:
    iqr_mask = iqr_outliers(df[col])
    z_mask = zscore_outliers(df[col])
    outlier_masks[col] = iqr_mask | z_mask
    outlier_rows.append({
        "Feature": col,
        "IQR outliers": int(iqr_mask.sum()),
        "Z-score outliers": int(z_mask.sum()),
        "Overlap": int((iqr_mask & z_mask).sum())
    })
display(pd.DataFrame(outlier_rows))

plt.figure(figsize=(8, 5))
sns.boxplot(x=df["GrLivArea"])
plt.title("GrLivArea outlier check")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "grlivarea_outliers_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", hue=outlier_masks["GrLivArea"], palette={False:"steelblue", True:"crimson"})
plt.title("GrLivArea vs SalePrice (IQR or Z-score flagged)")
plt.legend(title="Flagged")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outlier_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

print("Review flagged records before removing them. Some may be genuine unusual houses.")

## 9. Data quality checks and cleaning
Check duplicates and impossible values. In this dataset, many missing categorical values mean that a feature is absent (for example, no pool or no alley), not that the value is unknown. Keep the original `df` unchanged and create `df_clean` for a documented analysis-ready copy.

In [ ]:
duplicate_count = int(df.duplicated().sum())
print("Duplicate rows:", duplicate_count)

df_clean = df.drop_duplicates().copy()

# In these columns, missing often means the house does not have that feature.
absence_columns = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1",
    "BsmtFinType2", "MasVnrType"
]
for col in absence_columns:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

# Numeric missing values: use median; LotFrontage is better estimated within neighbourhood.
if "LotFrontage" in df_clean.columns:
    df_clean["LotFrontage"] = df_clean.groupby("Neighborhood")["LotFrontage"].transform(
        lambda s: s.fillna(s.median())
    )
    df_clean["LotFrontage"] = df_clean["LotFrontage"].fillna(df_clean["LotFrontage"].median())

for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Remaining categorical missing values: fill with the most frequent category.
for col in df_clean.select_dtypes(include=["object"]).columns:
    if df_clean[col].isna().any():
        mode_values = df_clean[col].mode()
        fill_value = mode_values.iloc[0] if not mode_values.empty else "Unknown"
        df_clean[col] = df_clean[col].fillna(fill_value)

print("Remaining missing values:", int(df_clean.isna().sum().sum()))
print("Cleaned shape:", df_clean.shape)
print("YearBuilt after YrSold:", int((df_clean["YearBuilt"] > df_clean["YrSold"]).sum()))
area_cols = ["LotArea", "GrLivArea", "TotalBsmtSF", "GarageArea"]
for col in area_cols:
    print(f"Negative values in {col}:", int((df_clean[col] < 0).sum()))

print("\\nCleaning note: filling missing feature labels with 'None' preserves the meaning that the feature is absent.")

## 10. Final dashboard and storytelling
Create a four-panel dashboard: average sale price by year built, highest-mean-price neighbourhoods, sale-price distribution with mean/median, and living area versus price coloured by quality.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

year_price = df.groupby("YearBuilt")["SalePrice"].mean().sort_index()
axes[0, 0].plot(year_price.index, year_price.values, label="Average price")
axes[0, 0].set_title("Average sale price by year built")
axes[0, 0].set_xlabel("Year built")
axes[0, 0].set_ylabel("Average sale price ($)")
axes[0, 0].legend()

neighborhood_mean = df.groupby("Neighborhood")["SalePrice"].mean().sort_values(ascending=False).head(10).sort_values()
neighborhood_mean.plot(kind="barh", ax=axes[0, 1])
axes[0, 1].set_title("Top 10 neighborhoods by mean sale price")
axes[0, 1].set_xlabel("Mean sale price ($)")
axes[0, 1].set_ylabel("Neighborhood")

sns.histplot(df["SalePrice"], kde=True, ax=axes[1, 0])
axes[1, 0].axvline(df["SalePrice"].mean(), linestyle="--", label="Mean")
axes[1, 0].axvline(df["SalePrice"].median(), linestyle=":", label="Median")
axes[1, 0].set_title("Sale price distribution")
axes[1, 0].set_xlabel("Sale price ($)")
axes[1, 0].legend()

sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", hue="OverallQual",
                palette="viridis", alpha=.7, ax=axes[1, 1], legend=False)
axes[1, 1].set_title("Living area vs sale price (colour = quality)")
axes[1, 1].set_xlabel("Above-ground living area (sq ft)")
axes[1, 1].set_ylabel("Sale price ($)")

fig.suptitle("House Price EDA Summary", fontsize=16)
plt.tight_layout()
dashboard_path = OUTPUT_DIR / "eda_dashboard.png"
plt.savefig(dashboard_path, dpi=180, bbox_inches="tight")
plt.show()
print("Dashboard saved to:", dashboard_path)

## 11. Conclusions
Use the calculated outputs above to support your presentation. The following summary is generated from the dataset rather than assumed in advance.

In [ ]:
mean_price = df["SalePrice"].mean()
median_price = df["SalePrice"].median()
print(f"Average sale price: ${mean_price:,.2f}")
print(f"Median sale price: ${median_price:,.2f}")
print(f"SalePrice skewness: {df['SalePrice'].skew():.3f}")
print("\\nTop 5 numerical correlations with SalePrice:")
display(corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False).head(5).to_frame("Absolute Pearson correlation"))
print("\\nTop 5 neighbourhoods by median sale price:")
display(df.groupby("Neighborhood")["SalePrice"].median().sort_values(ascending=False).head(5).to_frame("Median SalePrice"))
print("\\nPresentation reminder: correlation shows association, not proof that one feature causes a higher price.")

## Suggested mid-semester presentation
1. Introduce the project and objective.
2. Explain the Kaggle Ames Housing dataset (rows, columns, and target `SalePrice`).
3. Demonstrate data loading and missing-value inspection.
4. Explain numerical, categorical, and ordinal variables.
5. Show descriptive statistics and distribution plots.
6. Explain correlation, scatter plots, and the heatmap.
7. Explain why outliers are flagged and reviewed, not automatically deleted.
8. Describe missing-value decisions and data-quality checks.
9. Present the final dashboard and data-supported conclusions.

**Important:** This is exploratory analysis only. No machine-learning model or house-price prediction is built in this project.